In [1]:
import sys, os
current_dir = os.getcwd()
project_root = current_dir[:current_dir.find("src") - 1]
sys.path.insert(0, project_root)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.models.train_model.train_model import *

In [3]:
csv_semi_processed_path = os.path.join(project_root, "data", "processed", "semi_processed.csv")
df = pd.read_csv(csv_semi_processed_path, encoding='utf-8')

add_is_test_column(df, random_state=42)

save_model = True
save_model_folder = os.path.join(project_root, "src", "models", "fitted_models")
n_mimo = 4

df_r_selected = Data_selector(df).select_peaks(goodness=3)

base_features = ["name", "code", "temperature", "humidity", "dew", "surface_pressure", "value",
                    "forecast", "status"]
features_with_lag = ["temperature", "humidity", "dew", "surface_pressure"]
hours_delay = [1, 5]
lag_features = [f"{feature}_with_{hour}_delay" for feature in features_with_lag for hour in hours_delay]
# lag_features.append("generation_with_24_delay")
time_features = ["hour", "day_of_week", "month", "season", "datetime"]
df_f_selected = select_dataset_features(df_r_selected, base_features, lag_features, time_features, "generation")

model_X_cols = find_after_mimo_cols(df_f_selected, n_mimo)

train_indices = (df_r_selected['is_test'] == False)
train_df = df_f_selected[train_indices]
param={"n_est":20,"m_depth":10}
train_model(train_df, model_X_cols, n_mimo,param=param, save_model=save_model, save_model_folder=save_model_folder)

model = load_model(save_model_folder)

test_indices = (df_r_selected['is_test'] == True)
test_df = df_f_selected[test_indices]
test_model(model, test_df, model_X_cols, n_mimo)

2025-09-29 11:04:19 - train_model - INFO - Some features have been dropped successfully
2025-09-29 11:04:54 - train_model - INFO - Model has been trained successfully
2025-09-29 11:04:56 - train_model - INFO - Test rmse error: 4.062%
2025-09-29 11:04:56 - train_model - INFO - Test threshold error: 66.929%
2025-09-29 11:04:56 - train_model - INFO - Test rmae error: 2.709%
2025-09-29 11:04:56 - train_model - INFO - R2 score: 0.979%


In [10]:
def r_a(a):
    return " - ".join([f"{round(x,3):0.3f}" for x in a])
def show(error):
    for k in error:
        print(f"n_est {k[0]} m_depth {k[1]} => {r_a(error[k])}")
        
def check(params):
    error = {}
    for n_est in params["n_est"]:
        for m_depth in params["m_depth"]:
            param={"n_est":n_est,"m_depth":m_depth}
            rmse_error_train,thresh_error_train,rmae_error,r2_score = train_model(train_df, model_X_cols, n_mimo,param=param, save_model=save_model, save_model_folder=save_model_folder)
            error[(n_est,m_depth)] = (rmse_error_train,thresh_error_train,rmae_error,r2_score)
            print(f"n_est {n_est} m_depth {m_depth} => {r_a(error[(n_est,m_depth)])}")
    return error

In [ ]:
params={"n_est":[25,30,35,40],"m_depth":[10]}
error = check(params)
show(error)

2025-09-29 11:05:51 - train_model - INFO - Model has been trained successfully
2025-09-29 11:06:32 - train_model - INFO - Model has been trained successfully
2025-09-29 11:07:17 - train_model - INFO - Model has been trained successfully
2025-09-29 11:08:11 - train_model - INFO - Model has been trained successfully


n_est 25 m_depth 10 => 3.699 - 62.420 - 2.370 - 0.984
n_est 30 m_depth 10 => 3.700 - 62.397 - 2.366 - 0.984
n_est 35 m_depth 10 => 3.671 - 62.434 - 2.353 - 0.984
n_est 40 m_depth 10 => 3.665 - 62.381 - 2.350 - 0.984


In [8]:
params={"n_est":[15],"m_depth":[15,16,17,18,19,20]}
error = check(params)
show(error)

2025-09-29 11:10:53 - train_model - INFO - Model has been trained successfully
2025-09-29 11:11:27 - train_model - INFO - Model has been trained successfully
2025-09-29 11:12:03 - train_model - INFO - Model has been trained successfully
2025-09-29 11:12:39 - train_model - INFO - Model has been trained successfully
2025-09-29 11:13:14 - train_model - INFO - Model has been trained successfully
2025-09-29 11:13:52 - train_model - INFO - Model has been trained successfully


n_est 15 m_depth 15 => 1.922 - 37.191 - 1.188 - 0.996
n_est 15 m_depth 16 => 1.742 - 32.928 - 1.063 - 0.996
n_est 15 m_depth 17 => 1.582 - 28.971 - 0.954 - 0.997
n_est 15 m_depth 18 => 1.446 - 25.399 - 0.858 - 0.998
n_est 15 m_depth 19 => 1.339 - 22.587 - 0.782 - 0.998
n_est 15 m_depth 20 => 1.239 - 20.090 - 0.715 - 0.998


In [ ]:
params={"n_est":[15],"m_depth":range(20,40)}
error = check(params)
show(error)

2025-09-29 11:17:43 - train_model - INFO - Model has been trained successfully


n_est 15 m_depth 20 => 1.239 - 20.090 - 0.715 - 0.998


2025-09-29 11:18:24 - train_model - INFO - Model has been trained successfully


n_est 15 m_depth 21 => 1.164 - 17.844 - 0.663 - 0.998


2025-09-29 11:19:04 - train_model - INFO - Model has been trained successfully


...

n_est 15 m_depth 37 => 0.977 - 12.014 - 0.523 - 0.999


2025-09-29 11:30:05 - train_model - INFO - Model has been trained successfully


n_est 15 m_depth 38 => 0.979 - 12.011 - 0.523 - 0.999


2025-09-29 11:30:46 - train_model - INFO - Model has been trained successfully


n_est 15 m_depth 39 => 0.978 - 12.013 - 0.523 - 0.999
n_est 15 m_depth 20 => 1.239 - 20.090 - 0.715 - 0.998
n_est 15 m_depth 21 => 1.164 - 17.844 - 0.663 - 0.998
n_est 15 m_depth 22 => 1.109 - 16.456 - 0.625 - 0.999
n_est 15 m_depth 23 => 1.061 - 15.320 - 0.595 - 0.999
n_est 15 m_depth 24 => 1.027 - 14.258 - 0.570 - 0.999
n_est 15 m_depth 25 => 1.005 - 13.518 - 0.553 - 0.999
n_est 15 m_depth 26 => 0.995 - 12.998 - 0.543 - 0.999
n_est 15 m_depth 27 => 0.985 - 12.536 - 0.534 - 0.999
n_est 15 m_depth 28 => 0.981 - 12.254 - 0.530 - 0.999
n_est 15 m_depth 29 => 0.983 - 12.194 - 0.529 - 0.999
n_est 15 m_depth 30 => 0.974 - 12.017 - 0.525 - 0.999
n_est 15 m_depth 31 => 0.975 - 12.079 - 0.524 - 0.999
n_est 15 m_depth 32 => 0.978 - 12.048 - 0.524 - 0.999
n_est 15 m_depth 33 => 0.977 - 11.983 - 0.524 - 0.999
n_est 15 m_depth 34 => 0.979 - 11.998 - 0.523 - 0.999
n_est 15 m_depth 35 => 0.973 - 11.978 - 0.522 - 0.999
n_est 15 m_depth 36 => 0.973 - 12.042 - 0.523 - 0.999
n_est 15 m_depth 37 => 0.977